In [2]:
import os
%pwd


'/mnt/d/dl_project/research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'/mnt/d/dl_project'

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [20]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath =CONFIG_FILE_PATH,
                 params_filepath =PARAMS_FILE_PATH,
                 schema_filepath =SCHEMA_FILE_PATH
                 ):
        self.config =read_yaml(config_filepath)
        self.params =read_yaml(params_filepath)
        self.schema =read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        root_dir = Path(config.root_dir)
        source_url = config.source_URL
        local_data_file = Path(config.local_data_file)
        unzip_dir = Path(config.unzip_dir)

        create_directories([config.root_dir,local_data_file.parent,unzip_dir.parent])

        data_ingestion_config = DataIngestionConfig(
            root_dir = root_dir,
            source_URL =source_url,
            local_data_file = local_data_file,
            unzip_dir = unzip_dir )
        
        return data_ingestion_config


In [21]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [26]:
#data ingestion component
class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config

    def download_file(self) ->str:
        """
        Fetch data from url
        """
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = str(self.config.local_data_file)
            logger.info(f"downloading data from {dataset_url} into file {zip_download_dir}")

            file_id =dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?export=download&id="
            gdown.download(prefix+file_id,zip_download_dir,quiet =False)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    def extract_zip_file(self):
        """
        zip_file_path: str
        extracts the zip file into the data directory
        functions returns None
        """
        unzip_path = self.config.unzip_dir
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)



In [27]:
#data_ingestion_pipeline
try:
    config = ConfigurationManager()
    data_ingestion_config =config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


[2026-06-22 07:43:31,119: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-06-22 07:43:31,134: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-22 07:43:31,144: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-22 07:43:31,150: INFO: common: created directory at: artifacts]
[2026-06-22 07:43:31,160: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-22 07:43:31,170: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-22 07:43:31,175: INFO: common: created directory at: artifacts]
[2026-06-22 07:43:31,177: INFO: 3872451101: downloading data from https://drive.google.com/file/d/19JqEboAL3nefYVfA-s4dMBkwlqZPTuYR/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?id=19JqEboAL3nefYVfA-s4dMBkwlqZPTuYR
From (redirected): https://drive.google.com/uc?id=19JqEboAL3nefYVfA-s4dMBkwlqZPTuYR&confirm=t&uuid=5df7e81a-344d-4d42-bad3-a0498d5ddbf4
To: /mnt/d/dl_project/artifacts/data_ingestion/data.zip
100%|██████████| 1.63G/1.63G [06:12<00:00, 4.37MB/s]


[2026-06-22 07:49:50,225: INFO: 3872451101: Downloaded data from https://drive.google.com/file/d/19JqEboAL3nefYVfA-s4dMBkwlqZPTuYR/view?usp=sharing into file artifacts/data_ingestion/data.zip]
